## Ekstraksi fitur 
Dalam pemrosesan teks, ekstraksi fitur proses mengubah data teks yang sudah dibersihkan (setelah preprocessing) menjadi bentuk representasi numerik atau vektor agar dapat dipahami dan diolah oleh algoritma machine learning atau statistik.

Tujuan utama dalam ekstraksi fitur adalah:
1. Format matematis, komputer tidak membaca kata-kata seperti manusia;
2. Pengurangan kompleksitas, menyederhanakan data dengan mengambil atiribut atau variabel yang paling bermakna;
3. Mendukung analisis, menyimpan data untuk tugas klasifikasi, klastering, atau analisis sentimen 

## Library

In [32]:
import numpy as np
import polars as pl

## Dataset

In [33]:
df = pl.read_csv("data_twiter/hasil.csv")
df = df.select("tokens_lemmatized")

In [34]:
df

tokens_lemmatized
str
"""aneh prabowo tidak melantik wa…"
"""pernyataan rocky gerung wapres…"
"""wantimpres angkat bicara rocky…"
"""pembentukan wantimpres prabowo…"
"""orang berharga puluhan menteri…"
…
"""kasdam iii siliwangi brigjen t…"
"""menteri trenggono berharap fas…"
"""brmp sulut dampingi koperasi u…"


## Bag of Words
Proses untuk mengubah data teks (seperti kalimat atau dokumen) menjadi angka agar dapat diproses oleh algoritma.

### Code Tanpa Library

In [35]:
def bag_of_words(df: pl.DataFrame, text_col: str | None = None):
    text_col = text_col or df.columns[0]
    n_docs = df.height

    # Tokenisasi: pisah berdasarkan spasi, satu baris = satu token
    tokens = (
        df.select(pl.col(text_col).cast(pl.Utf8).fill_null("").str.split(" ").alias("token"))
        .with_row_index("doc_id")
        .explode("token")
        .filter(pl.col("token") != "")
    )

    # Hitung frekuensi kata per dokumen
    counts = tokens.group_by(["doc_id", "token"]).agg(pl.len().alias("tf"))

    # Vocabulary + indeks kolom
    vocab_df = counts.select("token").unique().sort("token").with_row_index("idx")
    counts = counts.join(vocab_df, on="token")

    # Isi matriks BoW
    X = np.zeros((n_docs, vocab_df.height), dtype=np.int32)
    X[counts["doc_id"].to_numpy(), counts["idx"].to_numpy()] = counts["tf"].to_numpy()

    return X, vocab_df["token"].to_list()

In [42]:
df2 = pl.DataFrame({
    "teks": [
        "Saya suka makan nasi goreng",
        "Nasi goreng itu enak sekali",
        "Saya tidak suka makan mie",
    ]
})


In [43]:
X, vocab = bag_of_words(df2, text_col="teks")


In [44]:
X

array([[0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0],
       [1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0],
       [0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1]], dtype=int32)

In [45]:
vocab

['Nasi',
 'Saya',
 'enak',
 'goreng',
 'itu',
 'makan',
 'mie',
 'nasi',
 'sekali',
 'suka',
 'tidak']

In [46]:
bow_df = pl.DataFrame(X, schema=vocab)
bow_df

Nasi,Saya,enak,goreng,itu,makan,mie,nasi,sekali,suka,tidak
i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32
0,1,0,1,0,1,0,1,0,1,0
1,0,1,1,1,0,0,0,1,0,0
0,1,0,0,0,1,1,0,0,1,1


### Code dengan Library

Kali ini coba menggunakan data dari hasil preprocessing sebelumnya

## N-Grams

## TF-IDF